# PHANTOM True Scratch - Custom Transformer Architecture

Train a cybersecurity LLM from **true zero** - custom transformer, random init!

**Runtime: GPU (T4 required)**
---


In [ ]:
# @title ## Setup
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!git clone https://github.com/Njap-png/Cogitrongit.git
%cd Cogitrongit

In [ ]:
# @title Install Dependencies
!pip install -q torch numpy

In [ ]:
# @title Prepare Cybersecurity Data
import json

qa_data = []
with open('data/phantom_train.jsonl', 'r') as f:
    for line in f:
        qa_data.append(json.loads(line))

# Add more cybersecurity content
cyber_corpus = """
PHANTOM is a cybersecurity AI assistant. PHANTOM helps people learn security. 
SQL injection attacks allow attackers to manipulate database queries. Use parameterized queries to prevent SQL injection. 
Cross-site scripting XSS injects malicious scripts into web pages. Sanitize user input to prevent XSS. 
Cross-site request forgery CSRF tricks users into submitting malicious requests. Use anti-CSRF tokens. 
Buffer overflow occurs when data exceeds memory boundaries. Use safe functions and stack canaries. 
Penetration testing finds vulnerabilities in systems. Always get written authorization first. 
OWASP Top 10 lists the most critical web security risks. 
CVE unique identifiers track publicly known vulnerabilities. 
Nmap is a network scanning tool. Use nmap to discover hosts and services. 
Metasploit is a penetration testing framework. 
Wireshark analyzes network traffic. 
Hashcat recovers passwords using GPU acceleration. 
Privilege escalation elevates access from user to admin. 
Ransomware encrypts files and demands payment. 
Phishing tricks users into revealing credentials. 
Network reconnaissance gathers information about targets. 
Incident response handles security breaches. 
Digital forensics recovers evidence from computers. 
Cloud security protects AWS Azure GCP environments. 
OSINT uses publicly available information. 
CTF competitions test security skills legally. 
Zero trust means never trust always verify. 
Defense in depth uses multiple security layers. 
Least privilege limits user access to what is needed. 
Input validation prevents malicious data entry. 
Encryption protects data at rest and in transit. 
Multi-factor authentication adds security layers. 
Security headers protect web applications. 
Vulnerability management finds and fixes flaws. 
Threat hunting proactively finds attackers. 
SIEM systems aggregate security logs. 
IDS detects suspicious network activity. 
IPS prevents detected attacks. 
Firewalls filter network traffic. 
VPN creates secure remote connections. 
MFA requires multiple verification methods. 
SSO allows one login for multiple apps. 
 SAML enables secure identity sharing. 
OAuth provides secure authorization. 
OpenID Connect is an identity layer on OAuth. 
JWT tokens transmit claims securely. 
HMAC verifies message integrity. 
RSA is an asymmetric encryption algorithm. 
AES is a symmetric encryption standard. 
SHA256 is a cryptographic hash function. 
TLS secures internet communications. 
SSL provides secure data transmission. 
Certificates verify identity cryptographically. 
PKI manages encryption keys. 
DNS translates domain names to IP addresses. 
DNSSEC adds security to DNS. 
ARP maps IP addresses to MAC addresses. 
ICMP is used for network diagnostics. 
TCP provides reliable data transfer. 
UDP is for fast unreliable transfer. 
HTTP is the web protocol. 
HTTPS adds SSL security to HTTP. 
FTP transfers files over networks. 
SSH provides secure remote access. 
SMTP sends email. 
POP3 retrieves email. 
IMAP accesses email remotely. 
SMB is for Windows file sharing. 
NFS is for Unix file sharing. 
LDAP is for directory services. 
Kerberos provides single sign-on. 
Active Directory manages Windows domains. 
Vulnerability scanning finds security flaws. 
Password cracking recovers passwords. 
Keylogging records keystrokes. 
Rootkits hide malicious software. 
Backdoors provide unauthorized access. 
Trojans masquerade as legitimate software. 
Viruses replicate and spread. 
Worms spread autonomously. 
Spyware monitors user activity. 
Adware displays unwanted ads. 
Botnets are networks of compromised computers. 
DDoS attacks overwhelm targets. 
DoS attacks deny service. 
MITM attacks intercept communications. 
Session hijacking takes over sessions. 
Cookie stealing steals session cookies. 
URL redirection redirects users maliciously. 
File inclusion exploits PHP files. 
XML external entity XXE attacks parsers. 
Denial of service stops legitimate access. 
Code injection executes malicious code. 
Command injection executes system commands. 
LDAP injection manipulates directory queries. 
XPath injection manipulates XML queries. 
Template injection exploits template engines. 
HTTP response splitting splits responses. 
HTTP request smuggling smuggles requests. 
Web cache poisoning poisons caches. 
Server-side request forgery SSRF accesses internal services. 
Insecure deserialization serializes untrusted data. 
Using components with known vulnerabilities is dangerous. 
Security misconfiguration weakens security. 
Missing function level access control exposes functions. 
Broken authentication allows unauthorized access. 
Sensitive data exposure leaks information. 
Lack of rate limiting allows attacks. 
CSRF tokens prevent cross-site requests. 
Content Security Policy prevents XSS. 
Secure headers protect web applications. 
CORS controls cross-origin access. 
Same-origin policy restricts cross-origin access. 
Subresource integrity verifies resources. 
HTTP Strict Transport Security enforces HTTPS. 
X-Frame-Options prevents clickjacking. 
X-Content-Type-Options prevents MIME sniffing. 
Referrer-Policy controls referrer information. 
Feature-Policy controls browser features. 
Permissions-Policy controls API access. 
PHANTOM combines five thinking engines for security analysis. 
PHANTOM uses chain of thought reasoning. 
PHANTOM uses parallel thinking from multiple perspectives. 
PHANTOM uses devil advocate to find flaws. 
PHANTOM uses meta cognition to monitor itself. 
PHANTOM uses red team to stress test conclusions. 
PHANTOM protects confidentiality. 
PHANTOM ensures integrity. 
PHANTOM maintains availability. 
PHANTOM helps prevent attacks. 
PHANTOM helps respond to incidents. 
PHANTOM teaches cybersecurity. 
PHANTOM is for authorized security research. 
PHANTOM follows ethical guidelines. 
PHANTOM promotes responsible disclosure. 
"""

# Format as instruction-response pairs
all_text = []
for item in qa_data:
    text = f"Question: {item['instruction']}\nAnswer: {item['output']}\n"
    all_text.append(text)

all_text.append(cyber_corpus)

full_text = "\n".join(all_text)
print(f"Corpus: {len(full_text)} chars, {len(all_text)} documents")

with open('/content/drive/MyDrive/phantom_data.txt', 'w') as f:
    f.write(full_text)
print("Saved to Drive")

In [ ]:
# @title Build Custom Transformer from Scratch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import math

class PHANTOMConfig:
    vocab_size = 5000
    max_len = 128
    n_layer = 6
    n_head = 6
    n_embd = 384  # Smaller for T4 GPU
    n_positions = 256
    dropout = 0.1

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=256):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class SelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head
        
        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)
        
    def forward(self, x, mask=None):
        B, T, C = x.shape
        qkv = self.qkv(x).split(self.n_embd * 3, dim=-1)
        q, k, v = [x.view(B, T, self.n_head, self.head_dim).transpose(1, 2) for x in qkv]
        
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            att = att.masked_fill(mask == 0, float('-inf'))
        att = torch.softmax(att, dim=-1)
        att = self.dropout(att)
        
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)

class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.attn = SelfAttention(config)
        self.ff = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout),
        )
    
    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ff(self.ln2(x))
        return x

class PHANTOMLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        self.token_emb = nn.Embedding(config.vocab_size, config.n_embd)
        self.pos_enc = PositionalEncoding(config.n_embd, config.n_positions)
        self.dropout = nn.Dropout(config.dropout)
        
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layer)])
        
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, x, targets=None):
        B, T = x.shape
        mask = torch.triu(torch.ones(T, T), diagonal=1).bool().to(x.device)
        
        x = self.token_emb(x)
        x = self.pos_enc(x)
        x = self.dropout(x)
        
        for block in self.blocks:
            x = block(x, mask)
        
        x = self.ln_f(x)
        logits = self.head(x)
        
        loss = None
        if targets is not None:
            loss = nn.CrossEntropyLoss()(logits.view(-1, self.config.vocab_size), targets.view(-1))
        
        return logits, loss

print("PHANTOM Transformer built from scratch!")
print(f"Parameters: {sum(p.numel() for p in PHANTOMConfig().__dict__.values())}")

In [ ]:
# @title Train Custom PHANTOM
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

# Simple tokenizer
class SimpleTokenizer:
    def __init__(self, text):
        chars = sorted(list(set(text)))
        self.stoi = {c: i+4 for i, c in enumerate(chars)}
        self.stoi['<pad>'] = 0
        self.stoi['<unk>'] = 1
        self.stoi['<s>'] = 2
        self.stoi['</s>'] = 3
        self.itos = {v: k for k, v in self.stoi.items()}
        self.vocab_size = len(self.stoi)
    
    def encode(self, s):
        return [self.stoi.get(c, 1) for c in s]
    
    def decode(self, ids):
        return ''.join(self.itos.get(i, '') for i in ids)

# Load corpus
with open('/content/drive/MyDrive/phantom_data.txt', 'r') as f:
    text = f.read()

tokenizer = SimpleTokenizer(text)
print(f"Vocabulary: {tokenizer.vocab_size} tokens")

# Dataset
class PHANTOMDataset(Dataset):
    def __init__(self, text, tokenizer, block_size=128):
        self.data = tokenizer.encode(text)
        self.block_size = block_size
        self.tokenizer = tokenizer
    
    def __len__(self):
        return len(self.data) - self.block_size
    
    def __getitem__(self, i):
        x = torch.tensor(self.data[i:i+self.block_size], dtype=torch.long)
        y = torch.tensor(self.data[i+1:i+self.block_size+1], dtype=torch.long)
        return x, y

config = PHANTOMConfig()
config.vocab_size = tokenizer.vocab_size

model = PHANTOMLM(config)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Using: {device}")

dataset = PHANTOMDataset(text, tokenizer)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

optimizer = optim.AdamW(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealLR(optimizer, T_max=500)

print("Starting training...")
for epoch in range(20):
    total_loss = 0
    for batch_idx, (x, y) in enumerate(loader):
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        logits, loss = model(x, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        if batch_idx % 10 == 0:
            print(f"Epoch {epoch} Batch {batch_idx}: Loss {loss.item():.4f}")
    
    scheduler.step()
    print(f"Epoch {epoch}: Avg Loss {total_loss/len(loader):.4f}\n")

# Save
torch.save({
    'model_state_dict': model.state_dict(),
    'config': config.__dict__,
    'tokenizer': tokenizer.__dict__,
}, '/content/drive/MyDrive/phantom_scratch.pt')
print("Model saved!")

In [ ]:
# @title Test PHANTOM
import torch

checkpoint = torch.load('/content/drive/MyDrive/phantom_scratch.pt')
config = PHANTOMConfig()
config.vocab_size = checkpoint['config']['vocab_size']
model = PHANTOMLM(config)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

tokenizer = checkpoint['tokenizer']
tokenizer = SimpleTokenizer('')  # Re-create
tokenizer.__dict__.update(checkpoint['tokenizer'])

device = next(model.parameters()).device

def generate(prompt, max_len=50):
    model.eval()
    with torch.no_grad():
        ids = tokenizer.encode(prompt)[:20]
        ids = torch.tensor([ids], dtype=torch.long).to(device)
        for _ in range(max_len):
            logits, _ = model(ids)
            next_token = logits[0, -1].argmax()
            ids = torch.cat([ids, next_token.unsqueeze(0).unsqueeze(0)], dim=1)
            if next_token.item() == 3:  # </s>
                break
        return tokenizer.decode(ids[0].cpu().tolist())

print("Testing PHANTOM:\n")
print("=" * 40)
result = generate("What is SQL")
print(result)
print("=" * 40)

---
## Done!

Custom PHANTOM transformer trained from scratch.
Expand data corpus for better results.